# End-to-End Feature Engineering Lab
### Dataset: Breast Cancer Wisconsin (Diagnostic) — UCI Machine Learning Repository

This lab walks through the **full feature engineering pipeline** a real project follows, on a genuine public dataset:

`sklearn.datasets.load_breast_cancer()` — 569 patients, 30 real-valued diagnostic measurements (cell nucleus features computed from digitized breast-mass images), target = malignant vs. benign. Source: W.N. Street, W.H. Wolberg, O.L. Mangasarian, UCI ML Repository (CC BY 4.0). It ships bundled with scikit-learn, so this notebook needs **no internet connection or file download** to run.

**Note on realism:** the original dataset is clean (no missing values, no categorical columns) because it's lab-grade diagnostic data. To practice the *full* range of feature engineering techniques — missing-value imputation, categorical encoding, datetime features — Section 2 deliberately injects a small amount of synthetic messiness (a `lab_batch` category and a `test_date` timestamp, plus some missing values). This is clearly marked wherever it happens; the underlying 30 diagnostic measurements and the malignant/benign target are always the real, unmodified data.

### Pipeline covered
0. Load & explore the real dataset
1. Simulate real-world messiness (marked synthetic)
2. Missing value imputation
3. Categorical encoding
4. Datetime feature extraction
5. Scaling & normalization
6. Outlier detection & treatment
7. Binning / discretization
8. Feature creation (interactions, ratios, polynomial terms)
9. Feature selection
10. Dimensionality reduction (PCA)
11. Full pipeline with `ColumnTransformer`
12. Does it work? Baseline vs. engineered-features model comparison


## 0. Load & Explore the Real Dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer

pd.set_option("display.max_columns", 15)
np.random.seed(42)

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target   # 0 = malignant, 1 = benign
df["diagnosis"] = df["target"].map({0: "malignant", 1: "benign"})

print(df.shape)
print(data.target_names)
df.head()

In [ ]:
df.describe().T[["mean", "std", "min", "max"]]

In [ ]:
df.info()
df.isna().sum().sum()   # confirm: genuinely zero missing values in the original data

### 📝 Try It
1. What proportion of patients are malignant vs. benign?
2. Which single raw feature has the strongest correlation with the target?


In [ ]:
# Your code here




### ✅ Solution

In [ ]:
print(df["diagnosis"].value_counts(normalize=True))

corrs = df.drop(columns=["target", "diagnosis"]).corrwith(df["target"]).abs().sort_values(ascending=False)
corrs.head(3)

## 1. Simulate Real-World Messiness *(synthetic — clearly marked)*

Real projects rarely hand you 30 clean numeric columns. We add:
- `lab_batch` — a synthetic categorical column (which lab batch processed the sample)
- `test_date` — a synthetic datetime column (when the sample was tested)
- Injected missing values in a few numeric columns, and a few extreme outliers

**Everything in this cell is synthetic scaffolding for the lab — the 30 real diagnostic measurements and the target label are untouched.**


In [ ]:
df_raw = df.copy()

# synthetic categorical column
df_raw["lab_batch"] = np.random.choice(["Batch_A", "Batch_B", "Batch_C"], size=len(df_raw), p=[0.5, 0.3, 0.2])

# synthetic datetime column across a 2-year window
df_raw["test_date"] = pd.to_datetime("2024-01-01") + pd.to_timedelta(
    np.random.randint(0, 730, size=len(df_raw)), unit="D"
)

# inject missing values into a few real numeric columns (simulating sensor/entry gaps)
for col in ["mean radius", "mean texture", "mean smoothness"]:
    missing_idx = df_raw.sample(frac=0.06, random_state=1).index
    df_raw.loc[missing_idx, col] = np.nan

# inject a handful of extreme outliers into 'mean area'
outlier_idx = df_raw.sample(5, random_state=2).index
df_raw.loc[outlier_idx, "mean area"] = df_raw.loc[outlier_idx, "mean area"] * np.random.uniform(4, 6, size=5)

print("Missing values now:\n", df_raw[["mean radius", "mean texture", "mean smoothness"]].isna().sum())
df_raw[["mean radius", "mean texture", "mean smoothness", "mean area", "lab_batch", "test_date"]].head()

## 2. Missing Value Imputation

In [ ]:
# Recap: three common strategies
from sklearn.impute import SimpleImputer, KNNImputer

num_cols_with_na = ["mean radius", "mean texture", "mean smoothness"]

# Strategy A: median imputation (robust to outliers, simple, usually the safe default)
median_imputer = SimpleImputer(strategy="median")
median_result = pd.DataFrame(median_imputer.fit_transform(df_raw[num_cols_with_na]), columns=num_cols_with_na)

# Strategy B: KNN imputation (uses similarity across other features — often more accurate)
knn_imputer = KNNImputer(n_neighbors=5)
knn_result = pd.DataFrame(knn_imputer.fit_transform(df_raw[num_cols_with_na]), columns=num_cols_with_na)

median_result.isna().sum(), knn_result.isna().sum()

### 📝 Try It
1. Add a `was_missing_mean_radius` indicator column (1 if the original value was missing, else 0) — this "missingness flag" pattern often carries real signal for tree-based models.
2. Impute `mean texture` and `mean smoothness` using the median strategy directly on `df_raw` (overwrite in place).


In [ ]:
# Your code here




### ✅ Solution

In [ ]:
df_raw["was_missing_mean_radius"] = df_raw["mean radius"].isna().astype(int)

imputer = SimpleImputer(strategy="median")
df_raw[["mean radius", "mean texture", "mean smoothness"]] = imputer.fit_transform(
    df_raw[["mean radius", "mean texture", "mean smoothness"]]
)

df_raw[["mean radius", "was_missing_mean_radius"]].head(8)

## 3. Categorical Encoding

In [ ]:
# Recap: one-hot vs. label vs. target (mean) encoding
one_hot = pd.get_dummies(df_raw["lab_batch"], prefix="batch")

from sklearn.preprocessing import LabelEncoder
label_encoded = LabelEncoder().fit_transform(df_raw["lab_batch"])

# target encoding — replace category with the mean target rate for that category
target_encoding_map = df_raw.groupby("lab_batch")["target"].mean()
target_encoded = df_raw["lab_batch"].map(target_encoding_map)

one_hot.head(), pd.Series(label_encoded).head().values, target_encoded.head()

### 📝 Try It
Merge the one-hot encoded `lab_batch` columns into `df_raw` and drop the original `lab_batch` column.

**Think first:** why is **one-hot** the right choice here rather than label encoding, given `lab_batch` has no natural order?


In [ ]:
# Your code here




### ✅ Solution

In [ ]:
df_raw = pd.concat([df_raw, one_hot], axis=1).drop(columns=["lab_batch"])
df_raw.filter(like="batch_").head()

# Why one-hot: label encoding (0,1,2) implies Batch_A < Batch_B < Batch_C, an ordering that
# doesn't exist in reality. A linear/distance-based model would wrongly treat that as meaningful.
# One-hot avoids inventing an order for a purely nominal category.

## 4. Datetime Feature Extraction

In [ ]:
# Recap
df_raw["test_month"] = df_raw["test_date"].dt.month
df_raw["test_dayofweek"] = df_raw["test_date"].dt.dayofweek
df_raw["test_is_weekend"] = df_raw["test_dayofweek"].isin([5, 6]).astype(int)
df_raw["days_since_start"] = (df_raw["test_date"] - df_raw["test_date"].min()).dt.days

df_raw[["test_date", "test_month", "test_dayofweek", "test_is_weekend", "days_since_start"]].head()

### 📝 Try It
Add a `test_quarter` column (1-4) and a cyclical encoding of `test_month` using sine/cosine
(cyclical encoding avoids treating December and January as "11 months apart" to a model).


In [ ]:
# Your code here




### ✅ Solution

In [ ]:
df_raw["test_quarter"] = df_raw["test_date"].dt.quarter
df_raw["month_sin"] = np.sin(2 * np.pi * df_raw["test_month"] / 12)
df_raw["month_cos"] = np.cos(2 * np.pi * df_raw["test_month"] / 12)

df_raw[["test_month", "test_quarter", "month_sin", "month_cos"]].head()

## 5. Scaling & Normalization

In [ ]:
# Recap — three scalers, three purposes
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

sample_cols = ["mean radius", "mean area", "mean smoothness"]

standard = StandardScaler().fit_transform(df_raw[sample_cols])   # mean=0, std=1 — most models
minmax   = MinMaxScaler().fit_transform(df_raw[sample_cols])      # squeezes into [0,1] — neural nets
robust   = RobustScaler().fit_transform(df_raw[sample_cols])      # uses median/IQR — resistant to outliers

pd.DataFrame(standard, columns=sample_cols).describe().loc[["mean", "std"]]

### 📝 Try It
`mean area` has injected outliers (Section 1). Compare `StandardScaler` vs. `RobustScaler` output
for that single column — which one keeps the bulk of the data in a tighter, more usable range?


In [ ]:
# Your code here




### ✅ Solution

In [ ]:
compare = pd.DataFrame({
    "standard_scaled": StandardScaler().fit_transform(df_raw[["mean area"]]).ravel(),
    "robust_scaled":  RobustScaler().fit_transform(df_raw[["mean area"]]).ravel()
})
compare.describe()
# RobustScaler's IQR-based scaling isn't dragged around by the 5 injected extreme outliers,
# so the bulk of normal values stay in a tighter, more model-friendly range.

## 6. Outlier Detection & Treatment

In [ ]:
# Recap — IQR method
Q1, Q3 = df_raw["mean area"].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outliers = df_raw[(df_raw["mean area"] < lower) | (df_raw["mean area"] > upper)]
print(f"{len(outliers)} outliers detected in 'mean area'")
outliers[["mean area"]]

### 📝 Try It
**Cap (winsorize)** `mean area` at the IQR bounds instead of dropping the rows — write the capped
values into a new column `mean_area_capped`.


In [ ]:
# Your code here




### ✅ Solution

In [ ]:
df_raw["mean_area_capped"] = df_raw["mean area"].clip(lower=lower, upper=upper)
df_raw.loc[outliers.index, ["mean area", "mean_area_capped"]]

## 7. Binning / Discretization

In [ ]:
# Recap
df_raw["radius_bucket"] = pd.cut(df_raw["mean radius"], bins=[0, 12, 15, 18, 30],
                                  labels=["Small", "Medium", "Large", "Very Large"])
df_raw["texture_quartile"] = pd.qcut(df_raw["mean texture"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

df_raw[["mean radius", "radius_bucket", "mean texture", "texture_quartile"]].head()

### 📝 Try It
Domain knowledge matters more than arbitrary bins: clinically, a `mean radius` under 15 is
associated with benign cases far more often. Create a 2-bucket `radius_risk_flag` at that
specific cutoff and check how it splits against `diagnosis`.


In [ ]:
# Your code here




### ✅ Solution

In [ ]:
df_raw["radius_risk_flag"] = np.where(df_raw["mean radius"] >= 15, "high_radius", "low_radius")
pd.crosstab(df_raw["radius_risk_flag"], df_raw["diagnosis"], normalize="index")

## 8. Feature Creation — Interactions, Ratios, Polynomial Terms

In [ ]:
# Recap — domain-driven ratio features often outperform raw measurements
df_raw["perimeter_to_radius"] = df_raw["mean perimeter"] / df_raw["mean radius"]
df_raw["area_to_perimeter"]  = df_raw["mean area"] / df_raw["mean perimeter"]
df_raw["concavity_x_compactness"] = df_raw["mean concavity"] * df_raw["mean compactness"]

df_raw[["perimeter_to_radius", "area_to_perimeter", "concavity_x_compactness"]].describe().loc[["mean","std"]]

In [ ]:
# Polynomial features (degree 2) on a small, deliberately chosen subset —
# NEVER blindly apply this to all 30 columns, it explodes combinatorially
from sklearn.preprocessing import PolynomialFeatures

subset = df_raw[["mean radius", "mean texture"]]
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(subset)
poly_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(subset.columns))
poly_df.head()

### 📝 Try It
Create a `worst_to_mean_radius` ratio (`worst radius` / `mean radius`) — a large ratio can indicate
an irregularly shaped, more concerning mass. Check its correlation with the target.


In [ ]:
# Your code here




### ✅ Solution

In [ ]:
df_raw["worst_to_mean_radius"] = df_raw["worst radius"] / df_raw["mean radius"]
df_raw["worst_to_mean_radius"].corr(df_raw["target"])

## 9. Feature Selection

In [ ]:
# Recap — three complementary filters
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

numeric_cols = df_raw.select_dtypes(include=[np.number]).drop(columns=["target"]).columns
X_numeric = df_raw[numeric_cols].fillna(0)
y = df_raw["target"]

# A. Low-variance filter — drop near-constant columns (they carry no information)
vt = VarianceThreshold(threshold=0.01)
vt.fit(X_numeric)
low_variance_cols = X_numeric.columns[~vt.get_support()]

# B. Mutual information — nonlinear relevance to the target
mi_scores = pd.Series(mutual_info_classif(X_numeric, y, random_state=0), index=X_numeric.columns)
top_mi = mi_scores.sort_values(ascending=False).head(8)

# C. Model-based importance
rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_numeric, y)
top_importance = pd.Series(rf.feature_importances_, index=X_numeric.columns).sort_values(ascending=False).head(8)

print("Low-variance columns to drop:", list(low_variance_cols))
top_mi

**Note on the low-variance list:** several genuinely useful columns show up there (e.g.
`mean smoothness`) simply because their raw values are naturally small (0.05-0.15 range) — variance
is scale-dependent. In practice, apply `VarianceThreshold` *after* scaling, or don't rely on it alone;
that's exactly why we cross-check against mutual information and model importance below.

### 📝 Try It
Compare the top 8 features from **mutual information** vs. the top 8 from **Random Forest
importance** — how much do the two lists overlap? (Different selection methods often agree on
the "obviously strong" features but disagree at the margins — that's expected and worth knowing.)


In [ ]:
# Your code here




### ✅ Solution

In [ ]:
overlap = set(top_mi.index) & set(top_importance.index)
print(f"{len(overlap)} of 8 features overlap:", overlap)
top_importance

## 10. Dimensionality Reduction — PCA

In [ ]:
# Recap — PCA needs scaled input first, always
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X_numeric)
pca = PCA(n_components=5, random_state=0)
X_pca = pca.fit_transform(X_scaled)

print("Explained variance by component:", np.round(pca.explained_variance_ratio_, 3))
print("Cumulative variance with 5 components:", round(pca.explained_variance_ratio_.sum(), 3))

### 📝 Try It
How many components are needed to explain **95%** of the variance? (Loop increasing
`n_components` and check the cumulative explained variance ratio.)


In [ ]:
# Your code here




### ✅ Solution

In [ ]:
pca_full = PCA(random_state=0).fit(X_scaled)
cumulative = np.cumsum(pca_full.explained_variance_ratio_)
n_needed = np.argmax(cumulative >= 0.95) + 1
print(f"{n_needed} components needed for 95% variance (out of {X_scaled.shape[1]} original features)")

## 11. Full Pipeline with `ColumnTransformer`

Everything above, assembled into one reusable, leak-free pipeline. This is what a real project
ships — not a notebook full of manual one-off steps.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

# Rebuild a slightly simplified raw frame for a clean pipeline demo
numeric_features = ["mean radius", "mean texture", "mean smoothness", "mean area", "mean perimeter"]
categorical_features = ["batch_A", "batch_B", "batch_C"] if "batch_A" in df_raw.columns else \
                        [c for c in df_raw.columns if c.startswith("batch_")]

X = df_raw[numeric_features + categorical_features]
y = df_raw["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", "passthrough", categorical_features)
])

full_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42))
])

full_pipeline.fit(X_train, y_train)
print("Test accuracy:", full_pipeline.score(X_test, y_test))

**Why this matters:** fitting the imputer/scaler *inside* the pipeline (rather than on the
whole dataset beforehand) means each cross-validation fold only ever sees its own training
statistics — this is what prevents **data leakage**, one of the most common real-world feature
engineering mistakes.

## 12. Capstone — Does Feature Engineering Actually Help?

The whole point of this lab: compare a model trained on **raw features only** vs. a model
trained on **engineered features**, using the same cross-validation setup.


### 📝 Try It
1. Build `X_baseline` from just the 30 original diagnostic columns (no engineering).
2. Build `X_engineered` including at least 3 engineered features from Sections 6-8
   (e.g. `mean_area_capped`, `perimeter_to_radius`, `worst_to_mean_radius`).
3. Run 5-fold cross-validation (`cross_val_score`, scoring="roc_auc") for both and compare mean AUC.


In [ ]:
# Your code here






### ✅ Solution

In [ ]:
from sklearn.model_selection import cross_val_score

original_cols = list(data.feature_names)
engineered_cols = original_cols + ["mean_area_capped", "perimeter_to_radius",
                                    "area_to_perimeter", "worst_to_mean_radius"]

X_baseline   = df_raw[original_cols].fillna(df_raw[original_cols].median())
X_engineered = df_raw[engineered_cols].fillna(df_raw[engineered_cols].median())
y = df_raw["target"]

model = RandomForestClassifier(n_estimators=300, random_state=42)

baseline_scores   = cross_val_score(model, X_baseline, y, cv=5, scoring="roc_auc")
engineered_scores = cross_val_score(model, X_engineered, y, cv=5, scoring="roc_auc")

print(f"Baseline  AUC: {baseline_scores.mean():.4f}  (+/- {baseline_scores.std():.4f})")
print(f"Engineered AUC: {engineered_scores.mean():.4f}  (+/- {engineered_scores.std():.4f})")

---
## Wrap-Up

You've run a genuinely complete feature engineering pipeline on a real public dataset:

`raw data → missingness/outlier handling → encoding → datetime features → scaling →
binning → feature creation → feature selection → PCA → leak-free pipeline → measurable model impact`

**Next steps:**
- Swap in a different bundled sklearn dataset (`load_wine()`, `load_diabetes()`) and repeat
  the whole notebook end-to-end on your own.
- Try `TargetEncoder`, `OneHotEncoder(handle_unknown="ignore")` inside the `ColumnTransformer`
  instead of pre-computed one-hot columns.
- Extend Section 12 into a full model comparison across Logistic Regression, Random Forest,
  and Gradient Boosting, with and without the engineered features.
